# 12 — Camera calibration: DLT for $P$, RQ decomposition, Zhang

**Needed for the assignment:** no (course / exam topic). **Study:** 3 h, **Practice:** 2 h.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## Idea

Calibration finds $K$ (intrinsics) and $R,\mathbf{t}$ (extrinsics). **With a 3D object** (known 3D points, for example a cube with a checkerboard): estimate $P$ by DLT, then split $P[:,1{:}3]=KR$ with an **RQ decomposition** (upper triangular times rotation). **With a planar chessboard** (Zhang): each image gives a homography $H=K[\mathbf{r}_1\ \mathbf{r}_2\ \mathbf{t}]$. Because $\mathbf{r}_1\perp\mathbf{r}_2$ and $\|\mathbf{r}_1\|=\|\mathbf{r}_2\|$, each $H$ gives 2 linear equations on $S=K^{-T}K^{-1}$. With 3 or more images you solve for $S$, then get $K$ from it.

## Key formulas

**DLT for $P$** (12 unknowns, 11 DoF, at least 6 points), two rows per point: $$[X,Y,Z,1,0,0,0,0,-uX,-uY,-uZ,-u],\qquad[0,0,0,0,X,Y,Z,1,-vX,-vY,-vZ,-v]$$ **Decomposition** ($P=KR[I\mid-\mathbf{t}]$): $\ KR=\text{RQ}(P_{3\times3})$,   $\mathbf{t}=-R^TK^{-1}\mathbf{p}_4$. Then rescale so that $K_{33}=1$, and flip signs (below) so that $\text{diag}(K)>0$ and $\det R=+1$.\
**RQ** (rq.pdf): multiply $A$ from the right by $R_x(\alpha),R_y(\beta),R_z(\gamma)$ so that entries $(3,2)$, $(3,1)$, $(2,1)$ become $0$: $$\tan\alpha=-\tfrac{a_{32}}{a_{33}},\quad\tan\beta=\tfrac{b_{31}}{b_{33}},\quad\tan\gamma=-\tfrac{c_{21}}{c_{22}},\qquad
A R_xR_yR_z=D\ \Rightarrow\ A=D\,(R_xR_yR_z)^T$$ **Sign fix:** $D Q=(DS)(SQ)$ for $S=\text{diag}(\pm1)$, since $S^2=I$.\
**Zhang:** $\mathbf{h}_1^TS\mathbf{h}_2=0$ and $\mathbf{h}_1^TS\mathbf{h}_1-\mathbf{h}_2^TS\mathbf{h}_2=0$. Solve for $S$ (6 unknowns) with SVD, then use the closed form for $K$. Extrinsics: $[\mathbf{r}_1\ \mathbf{r}_2\ \mathbf{t}]=\lambda K^{-1}H$ with $\lambda=1/\|K^{-1}\mathbf{h}_1\|$, and $\mathbf{r}_3=\mathbf{r}_1\times\mathbf{r}_2$.

# Worked example (on paper): RQ by Givens rotations

$M=\begin{bmatrix}2&1&0\\0&1&-2\\0&1&0\end{bmatrix}$ (this is $K R_x(90^\circ)$ with $K=\begin{bmatrix}2&0&1\\0&2&1\\0&0&1\end{bmatrix}$; we want to recover $K$).

**1. Zero $(3,2)$.** $a_{32}=1$, $a_{33}=0$, so $\alpha=90^\circ$ and $R_x=\begin{bmatrix}1&0&0\\0&0&-1\\0&1&0\end{bmatrix}$: $$B=MR_x=\begin{bmatrix}2&0&-1\\0&-2&-1\\0&0&-1\end{bmatrix}$$ (the columns of $B$ are $M\mathbf{e}_1$, $M\mathbf{e}_3$ and $-M\mathbf{e}_2$).

**2.** $b_{31}=0\Rightarrow\beta=0$, and $c_{21}=0\Rightarrow\gamma=0$. So $D=B$ is already upper triangular, and $Q=R_x^T$.

**3. Sign fix.** $D$ has diagonal $(2,-2,-1)$, so use $S=\text{diag}(1,-1,-1)$: $$K=DS=\begin{bmatrix}2&0&1\\0&2&1\\0&0&1\end{bmatrix},\qquad R=SR_x^T=\begin{bmatrix}1&0&0\\0&0&-1\\0&1&0\end{bmatrix}=R_x(90^\circ)\ ✓,\quad\det R=1.$$

# The same in Python / NumPy (RQ from rq.pdf, then a full calibration on synthetic data)

In [2]:
import numpy as np

def Rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])
def Ry(b):
    return np.array([[np.cos(b), 0, np.sin(b)], [0, 1, 0], [-np.sin(b), 0, np.cos(b)]])
def Rz(g):
    return np.array([[np.cos(g), -np.sin(g), 0], [np.sin(g), np.cos(g), 0], [0, 0, 1]])

def rq(A):
    """RQ decomposition as in rq.pdf: A Rx Ry Rz = D (upper triangular), A = D (Rx Ry Rz)^T"""
    a = np.arctan2(-A[2, 1], A[2, 2]);  B = A @ Rx(a)      # zero at (3,2)
    b = np.arctan2(B[2, 0], B[2, 2]);   C = B @ Ry(b)      # zero at (3,1)
    g = np.arctan2(-C[1, 0], C[1, 1]);  D = C @ Rz(g)      # zero at (2,1)
    Q = (Rx(a) @ Ry(b) @ Rz(g)).T
    # make the diagonal of K positive: D Q = (D S)(S Q) with S = diag(+-1)
    S = np.diag(np.sign(np.diag(D)))
    return D @ S, S @ Q

# --- paper example ---
M = np.array([[2.0, 1, 0], [0, 1, -2], [0, 1, 0]])
K, R = rq(M)
print("K =\n", np.round(K, 6), "\nR =\n", np.round(R, 6))
print("K R == M ?", np.allclose(K @ R, M), "  det R =", round(np.linalg.det(R), 6))

# --- full calibration by a 3D object (DLT for P) on synthetic data ---
K_true = np.array([[800, 0, 320], [0, 780, 240], [0, 0, 1.0]])
R_true = Ry(np.radians(20)) @ Rx(np.radians(-10))
C_true = np.array([0.5, -0.3, -4.0])                       # camera centre (= t of lecture)
P_true = K_true @ R_true @ np.hstack([np.eye(3), -C_true.reshape(3, 1)])

rng = np.random.default_rng(0)
X = rng.uniform(-1, 1, size=(20, 3))                       # points of a "calibration cube"
x = (np.column_stack([X, np.ones(20)]) @ P_true.T)
u = x[:, :2] / x[:, 2:3] + rng.normal(0, 0.3, size=(20, 2))

A = []
for (Xw, Yw, Zw), (ui, vi) in zip(X, u):
    A.append([Xw, Yw, Zw, 1, 0, 0, 0, 0, -ui * Xw, -ui * Yw, -ui * Zw, -ui])
    A.append([0, 0, 0, 0, Xw, Yw, Zw, 1, -vi * Xw, -vi * Yw, -vi * Zw, -vi])
A = np.array(A)
val, vec = np.linalg.eig(A.T @ A)
P = vec.real[:, np.argmin(val.real)].reshape(3, 4)

K, R = rq(P[:, :3])
scale = K[2, 2]
K, P = K / scale, P / scale                                # K[2,2] = 1
if np.linalg.det(R) < 0:                                   # P is defined up to sign
    R, P = -R, -P
C = -R.T @ np.linalg.inv(K) @ P[:, 3]                      # t = -R^T K^-1 p4
print("K estimated =\n", np.round(K, 1))
print("camera centre estimated =", np.round(C, 3), " true =", C_true)

K =
 [[2. 0. 1.]
 [0. 2. 1.]
 [0. 0. 1.]] 
R =
 [[ 1.  0.  0.]
 [ 0.  0. -1.]
 [ 0.  1.  0.]]
K R == M ? True   det R = 1.0
K estimated =
 [[795.7  -0.6 316.7]
 [ -0.  778.5 241.2]
 [  0.    0.    1. ]]
camera centre estimated = [ 0.499 -0.298 -3.991]  true = [ 0.5 -0.3 -4. ]


## Course / lab link

The Practice notebook uses `scipy.linalg.rq` and gets a $K$ with **negative diagonal** entries and $K_{33}\neq1$: the sign fix and rescaling above are missing. In its Zhang part, $\mathbf{r}_1$ has norm $0.545$: the factor $\lambda=1/\|K^{-1}\mathbf{h}_1\|$ is missing, so $\mathbf{r}_3$ and $\mathbf{t}$ are scaled wrong. The code above uses $\mathbf{t}=-R^TK^{-1}\mathbf{p}_4$ (lecture convention); the notebook's $\mathbf{t}=K^{-1}\mathbf{p}_4$ is the other convention ($\mathbf{t}'=-R\mathbf{t}$).

## Practice

Write your solution in the empty cell under each exercise. The **expected result** tells you what you should get, not how to get it. Grey *provided data* cells are already filled in; run them first.

### Exercise 1

Do RQ by hand (rq.pdf steps) for $M=KR_z(90^\circ)$ with $K=\begin{bmatrix}2&0&1\\0&2&1\\0&0&1\end{bmatrix}$. Which angle is not zero?

> **Expected result:** $M=\begin{bmatrix}0&-2&1\\2&0&1\\0&0&1\end{bmatrix}$. $\alpha=\beta=0$ and $\gamma=-90^\circ$; then $D=K$ (no sign fix needed) and $R=R_z(90^\circ)$.

In [3]:
# Exercise 1: your solution


### Exercise 2

Estimate $P$ by DLT from the provided 10 correspondences and compute the reprojection error. What is the minimum number of points?

> **Expected result:** Max reprojection error $\approx10^{-8}$ px (noise-free data). Minimum: 6 points (11 unknowns, 2 equations per point).

In [4]:
# --- provided data (run me first) ---
# Xw : (10, 3) world points of a calibration object, uniform in [-1, 1]^3
# uv : (10, 2) their exact pixels (no noise); first row ~ [111.74, 73.69]
# K_true, R_true, C_true are only here to make the data (to compare with at the end).
def Rx(a): return np.array([[1,0,0],[0,np.cos(a),-np.sin(a)],[0,np.sin(a),np.cos(a)]])
def Ry(b): return np.array([[np.cos(b),0,np.sin(b)],[0,1,0],[-np.sin(b),0,np.cos(b)]])
def Rz(g): return np.array([[np.cos(g),-np.sin(g),0],[np.sin(g),np.cos(g),0],[0,0,1]])
rng = np.random.default_rng(11)
K_true = np.array([[900, 0, 400], [0, 880, 300], [0, 0, 1.0]])
R_true = Rz(np.radians(5)) @ Ry(np.radians(-15)) @ Rx(np.radians(10))
C_true = np.array([-0.4, 0.2, -5.0])
P_true = K_true @ R_true @ np.hstack([np.eye(3), -C_true.reshape(3, 1)])
Xw = rng.uniform(-1, 1, size=(10, 3))
x = np.column_stack([Xw, np.ones(10)]) @ P_true.T
uv = x[:, :2] / x[:, 2:3]

In [5]:
# Exercise 2: your solution


### Exercise 3

Decompose your $P$ with your own `rq` (sign fix, $K_{33}=1$, $\det R=+1$) and recover the camera centre $\mathbf t=-R^TK^{-1}\mathbf p_4$.

> **Expected result:** $K=\begin{bmatrix}900&0&400\\0&880&300\\0&0&1\end{bmatrix}$, $R=$ `R_true`, centre $=(-0.4,\ 0.2,\ -5.0)$.

In [6]:
# Exercise 3: your solution


### Exercise 4

Why does $P$ have 11 DoF and not 12? Put all 10 points on the plane $Z=0$, build $A$ again, and count the (near) zero eigenvalues of $A^TA$.

> **Expected result:** $P$ is only defined up to scale, so 12 − 1 = 11. With planar points there are **4** eigenvalues $\approx0$ (relative size $<10^{-15}$): the three $Z$-columns are all zero, so the solution is not unique.

In [7]:
# Exercise 4: your solution
